# Chess Coach Models: the lab's runs on a GPU

Runs the lab's own commands on this notebook's GPU: for each model, its dataset from open Lichess
data, its runs, and its leaderboard. Choose a GPU runtime (Runtime, Change runtime type), set the
first cell, and run every cell.

The last cell packs the results into `/content/lab-results.zip`. **Save that file from the Files
panel before the runtime is released**: a runtime's disk goes with it. Unzip it at the root of a
clone of the repository, then `lab eval <model>` for each model, `lab report`, commit and push.

The notebook holds no credentials and pushes nothing. `docs/compute.md` in the repository says
what runs where.

In [ ]:
# The models to run, and for each the runs to train: "" trains every config under
# models/<model>/configs/. Name a few to keep a session short, with gbm-hand among them:
# every other run is compared with it, on this machine.
MODELS = ["puzzle-difficulty", "blunder-hazard", "think-time"]
RUNS = ""
REF = "main"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc && free -g | head -2

In [ ]:
%cd /content
!rm -rf chess-coach-models
!git clone --quiet --depth 1 --branch {REF} https://github.com/Aneesh-Pothuru/chess-coach-models.git
%cd /content/chess-coach-models
!git log --oneline -1
# Colab ships Torch with CUDA. The lab's other needs are small, and --no-deps keeps Torch as it is.
!pip install -q python-chess zstandard lightgbm polars pyyaml
!pip install -q --no-deps -e .
!python -c "import torch, lightgbm, polars, chess, zstandard; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# First a check that every run works on this GPU: each model end to end on a small sample,
# a few minutes in all. It writes under artifacts/smoke/ and is never a result.
shell = get_ipython()
for model in MODELS:
    shell.system(f"lab all {model} --smoke")
    if shell.user_ns.get("_exit_code", 0) != 0:
        raise SystemExit(f"the smoke run of {model} failed: fix that before spending GPU time on the full runs")

In [ ]:
import os

os.makedirs("artifacts/logs", exist_ok=True)
shell = get_ipython()
for model in MODELS:
    command = f"lab all {model}" + (f" --runs {RUNS}" if RUNS else "")
    print(f"\n=== {command}", flush=True)
    shell.system(f"bash -c 'set -o pipefail; {command} 2>&1 | tee artifacts/logs/{model}.log'")
    if shell.user_ns.get("_exit_code", 0) != 0:
        raise SystemExit(f"{command} failed: read artifacts/logs/{model}.log")

In [ ]:
from pathlib import Path

for model in MODELS:
    print(Path(f"models/{model}/results/leaderboard.md").read_text())

In [ ]:
!rm -f /content/lab-results.zip
!zip -q -r /content/lab-results.zip models/*/results artifacts/logs
!ls -la /content/lab-results.zip && sha256sum /content/lab-results.zip
print("LAB RUNS DONE: save /content/lab-results.zip from the Files panel now")